# 03 - Entrenamiento de Faster R-CNN para detección de humo y fuego

Detector de dos etapas (`fasterrcnn_resnet50_fpn_v2` de torchvision) preentrenado
en COCO, con el cabezal ajustado a las clases `smoke` y `fire`.

El código reusable vive en `src/`; este notebook solo arma la configuración,
corre el bucle de épocas y delega el reporte.

## Dónde correrlo

El notebook detecta el entorno y se adapta solo: corre en **Colab**, en **Kaggle
Notebooks** o en una **GPU local**.

La corrida versionada entrenó las 30 épocas enteras en la T4 de Kaggle. Como
son más de 18 horas y una sesión de Kaggle corta a las 12, se repartió entre
varias sesiones reanudando desde el checkpoint; el total de épocas del YAML no
cambió en el camino, así que el coseno del learning rate baja de forma continua
de punta a punta.

### Cómo seguir en Kaggle

1. Notebook nuevo, panel derecho **Session options > Accelerator > GPU T4 x2**
   e **Internet > On**. Ambas opciones aparecen recién cuando la cuenta está
   verificada por teléfono. No elegir la P100: es `sm_60` y el PyTorch que trae
   la imagen actual de Kaggle solo incluye kernels desde `sm_70`.
2. **Add Input > Datasets** y buscar `sayedgamal99/smoke-fire-detection-yolo`.
3. Subir `last_checkpoint.pth` como Dataset privado y agregarlo también con
   **Add Input**. La celda de recuperación lo copia a `/kaggle/working` y el
   entrenamiento sigue desde donde quedó.
4. Correr todo. Antes de las 12 h, **Save Version > Quick Save**: conserva
   `/kaggle/working` como output de la versión. **Save & Run All** no sirve acá,
   porque reejecuta el notebook desde cero en un contenedor limpio.
5. Para cada sesión siguiente, **Add Input > Your Work** y elegir la última
   versión guardada, en vez de volver a agregar el Dataset del checkpoint
   inicial: la celda de recuperación prioriza el checkpoint más reciente entre
   todos los inputs montados, así que agregar el output alcanza para seguir
   donde quedó esta sesión sin perder las épocas ya entrenadas.
6. Al terminar, la última celda arma un `.zip` con los artefactos y `last.pth`
   para bajar, commitear en el repo y subir a Drive.

## Salidas esperadas

En `reports/results/fasterrcnn_r50fpn/`: `results.csv`, `results.png`,
`confusion_matrix.png`, `confusion_matrix_normalized.png`, `PR_curve.png`,
`F1_curve.png`, `P_curve.png`, `R_curve.png`, `experiment_config_used.yaml`
y `metrics_summary.csv`.

In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import sys
import random
import shutil
import time
import yaml

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
# KAGGLE_KERNEL_RUN_TYPE lo define el runtime de Kaggle y no existe si alguien
# instala el paquete `kaggle` en otra máquina, a diferencia de /kaggle o de la
# variable KAGGLE_URL_BASE que trae la librería.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    ENV = "colab"
elif IN_KAGGLE:
    ENV = "kaggle"
else:
    ENV = "local"

print("Entorno:", ENV)
print("Directorio actual:", Path.cwd())

In [ ]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
# Rama del repositorio desde la que se clona y a la que se commitean los
# resultados. Mientras el PR este abierto tiene que apuntar a la rama del PR;
# una vez mergeado, cambiar a "main".
REPO_BRANCH = "feat/modelos-adicionales-deteccion"

RAW_REQUIREMENTS = (
    "https://raw.githubusercontent.com/Gabriela-Sol/"
    f"{REPO_NAME}/{REPO_BRANCH}/requirements.txt"
)

if IN_COLAB:
    !pip install -q -r {RAW_REQUIREMENTS}
elif IN_KAGGLE:
    # Solo lo que falta en la imagen: torch, torchvision, PIL, matplotlib,
    # numpy, pandas y yaml ya vienen. requirements.txt lista `torch` sin pinear
    # la build de CUDA, y dejar que pip lo resuelva en Kaggle reemplazaría el
    # torch preinstalado (que sí viene compilado contra el CUDA de la imagen)
    # por uno cualquiera de PyPI.
    !pip install -q torchmetrics pycocotools

print("Dependencias instaladas.")

In [ ]:
# ============================================================
# Verificación de GPU
# ============================================================

import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
    print("GPU:", DEVICE_NAME)

    # La P100 de Kaggle es sm_60 y el PyTorch de la imagen ya no trae kernels
    # por debajo de sm_70. CUDA igual reporta la placa como disponible, así que
    # sin este corte el error aparece recién en el primer forward, después de
    # haber esperado todo el setup y la carga del dataset.
    def _sm_a_tupla(arch: str) -> tuple[int, int]:
        numero = arch.removeprefix("sm_")
        # Las variantes "family-specific" (sm_90a, sm_100a, sm_120a) agregan
        # una letra al final para builds con instrucciones extra que no son
        # binario-compatibles con el resto de la familia; para decidir el
        # mínimo soportado alcanza con la parte numérica.
        numero = numero.rstrip("abcdefghijklmnopqrstuvwxyz")
        return int(numero[:-1]), int(numero[-1])

    soportadas = [
        _sm_a_tupla(arch)
        for arch in torch.cuda.get_arch_list()
        if arch.startswith("sm_")
    ]
    capacidad = torch.cuda.get_device_capability(0)

    # Se compara contra el mínimo y no con `in soportadas` porque el binario de
    # una sm es compatible hacia arriba dentro de la misma major (sm_86 corre en
    # sm_89): la lista no enumera todas las placas que efectivamente funcionan.
    if soportadas and capacidad < min(soportadas):
        minima = "sm_{}{}".format(*min(soportadas))
        raise RuntimeError(
            f"{DEVICE_NAME} es sm_{capacidad[0]}{capacidad[1]} y este PyTorch "
            f"solo soporta desde {minima}. Elegir otra GPU: en Kaggle, "
            f"Session options > Accelerator > GPU T4 x2."
        )
else:
    DEVICE = torch.device("cpu")
    DEVICE_NAME = "cpu"
    if IN_COLAB:
        # Cortar acá y no avisar nomás: en CPU la corrida no termina nunca y el
        # usuario se enteraría recién dentro de varias horas.
        raise RuntimeError(
            "No se detectó GPU. Faster R-CNN en CPU es inviable para 30 épocas. "
            "Activar Entorno de ejecución > Cambiar tipo de entorno > GPU."
        )
    if IN_KAGGLE:
        raise RuntimeError(
            "No se detectó GPU. Faster R-CNN en CPU es inviable para 30 épocas. "
            "Activar Session options > Accelerator > GPU T4 x2."
        )
    print("Sin GPU y fuera de Colab: se sigue en CPU, solo sirve para pruebas cortas.")

print("Device:", DEVICE)

In [ ]:
# ============================================================
# Almacenamiento de trabajo (Drive en Colab)
# ============================================================
# WORK_DIR es la raíz escribible del entorno: de ahí sale el clon del repo.
# RUNS_DIR es donde viven los checkpoints y los pesos, y tiene que persistir
# entre sesiones: en Colab eso es Drive, en Kaggle es /kaggle/working más un
# Quick Save.

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
    WORK_DIR = Path("/content")
    RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
    print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
elif IN_KAGGLE:
    # Único directorio escribible que Kaggle preserva al hacer Quick Save, y por
    # lo tanto el único desde el que se puede reanudar en otra sesión.
    WORK_DIR = Path("/kaggle/working")
    RUNS_DIR = WORK_DIR / "runs"
else:
    WORK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUNS_DIR = WORK_DIR / "runs"

RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("WORK_DIR:", WORK_DIR)
print("Carpeta de corridas:", RUNS_DIR)

In [ ]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

if ENV == "local":
    # Ya estamos dentro del repo: no hay nada que clonar.
    PROJECT_DIR = WORK_DIR
else:
    PROJECT_DIR = WORK_DIR / REPO_NAME

    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git checkout {REPO_BRANCH}
        !git pull origin {REPO_BRANCH}
    else:
        print("Clonando repositorio...")
        %cd {WORK_DIR}
        !git clone -b {REPO_BRANCH} {REPO_URL}.git
        %cd {PROJECT_DIR}

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Contenido del proyecto:", os.listdir(PROJECT_DIR))

In [ ]:
# ============================================================
# Carga de configuración del experimento
# ============================================================

EXPERIMENT_CONFIG_PATH = PROJECT_DIR / "configs" / "experiments" / "fasterrcnn_r50fpn.yaml"

if not EXPERIMENT_CONFIG_PATH.exists():
    raise FileNotFoundError(f"No se encontró la configuración: {EXPERIMENT_CONFIG_PATH}")

with open(EXPERIMENT_CONFIG_PATH, "r", encoding="utf-8") as file:
    experiment_config = yaml.safe_load(file)

experiment_name = experiment_config["experiment"]["name"]
training_cfg = experiment_config["training"]

print("Experimento:", experiment_name)
print("Familia:", experiment_config["experiment"]["family"])
print("Modelo:", experiment_config["experiment"]["model"])
print("Épocas:", training_cfg["epochs"], "| batch:", training_cfg["batch"])

In [ ]:
# ============================================================
# Descarga o localización del dataset
# ============================================================

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"


def find_yolo_dataset_dir(root: Path) -> Path:
    """Busca la carpeta con train/images, train/labels, val/images y val/labels."""
    for candidate in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all(
            (candidate / split / kind).exists()
            for split in ["train", "val"]
            for kind in ["images", "labels"]
        ):
            return candidate
    raise FileNotFoundError("No se encontró una estructura YOLO válida.")


DATA_DIR = None

if IN_KAGGLE:
    # El dataset ya está publicado en Kaggle, así que si se agregó con
    # Add Input > Datasets se monta en /kaggle/input y no hay nada que bajar.
    for mount in sorted(Path("/kaggle/input").glob("*")):
        if not mount.is_dir():
            continue
        try:
            DATA_DIR = find_yolo_dataset_dir(mount)
        except FileNotFoundError:
            continue
        print("Dataset montado desde los inputs del notebook:", mount.name)
        break
    if DATA_DIR is None:
        print(
            f"No se encontró el dataset en /kaggle/input. Agregarlo con "
            f"Add Input > Datasets > {DATASET_ID} para evitar la descarga."
        )

if DATA_DIR is None:
    import kagglehub

    DATA_DIR = find_yolo_dataset_dir(Path(kagglehub.dataset_download(DATASET_ID)))

print("Carpeta de datos:", DATA_DIR)

In [ ]:
# ============================================================
# Datasets y dataloaders
# ============================================================

from torch.utils.data import DataLoader

from src.data.yolo_dataset import YoloDetectionDataset, collate_fn

train_dataset = YoloDetectionDataset(DATA_DIR / "train", train=True, hflip_prob=0.5, seed=SEED)
val_dataset = YoloDetectionDataset(DATA_DIR / "val", train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=training_cfg["batch"],
    shuffle=True,
    num_workers=training_cfg["workers"],
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=training_cfg["batch"],
    shuffle=False,
    num_workers=training_cfg["workers"],
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

print("Imágenes de train:", len(train_dataset))
print("Imágenes de val:  ", len(val_dataset))

# Casi la mitad de las imágenes de train son negativos deliberados.
sin_cajas = sum(1 for i in range(200) if len(train_dataset[i][1]["boxes"]) == 0)
print(f"Negativos en las primeras 200 imágenes: {sin_cajas}")

In [ ]:
# ============================================================
# Modelo, optimizador y scheduler
# ============================================================

from src.engine.trainer import build_optimizer, build_scheduler
from src.modeling.detectors import build_fasterrcnn, count_parameters

torch.manual_seed(SEED)

model = build_fasterrcnn(
    num_classes=3,  # fondo + smoke + fire
    backbone=training_cfg["backbone"],
    trainable_backbone_layers=training_cfg["trainable_backbone_layers"],
    min_size=training_cfg["imgsz"],
    max_size=training_cfg["max_size"],
    pretrained=True,
)
model.to(DEVICE)

optimizer = build_optimizer(model, training_cfg)
scheduler = build_scheduler(optimizer, training_cfg, training_cfg["epochs"])
scaler = torch.amp.GradScaler("cuda") if (training_cfg["amp"] and DEVICE.type == "cuda") else None

print(f"Parámetros entrenables: {count_parameters(model) / 1e6:.2f} M")
print("Optimizador:", type(optimizer).__name__, "| scheduler:", type(scheduler).__name__)
print("AMP:", scaler is not None)

In [ ]:
# ============================================================
# Recuperar la corrida de una sesión anterior (Kaggle)
# ============================================================
# Kaggle borra /kaggle/working al cerrar la sesión. El checkpoint sobrevive de
# dos formas y las dos se montan igual en /kaggle/input: como Dataset privado
# subido a mano, por ejemplo uno traído de otra plataforma, o como output de una
# versión guardada con Quick Save y remontada con Add Input > Your Work.
#
# En Colab no hace falta: la corrida vive en Drive, que persiste entre sesiones.

from src.data.kaggle_inputs import restore_checkpoint_from_inputs

CHECKPOINT_PATH = RUNS_DIR / experiment_name / "last_checkpoint.pth"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

if IN_KAGGLE:
    recuperado = restore_checkpoint_from_inputs(Path("/kaggle/input"), CHECKPOINT_PATH)
    if recuperado is None:
        print("No hay checkpoint en /kaggle/input: el entrenamiento arranca de cero.")
    else:
        print("Checkpoint recuperado en:", recuperado)

In [ ]:
# ============================================================
# Reanudar desde el último checkpoint si existe
# ============================================================

from src.engine.trainer import load_checkpoint

start_epoch = 0
history = []

if CHECKPOINT_PATH.exists():
    start_epoch, history = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, DEVICE)
    print(f"Checkpoint encontrado. Reanudando desde la época {start_epoch + 1}.")

    # El checkpoint también restaura el estado del scheduler, incluido el total
    # de épocas con el que se creó (T_max del coseno). Si después de esa corrida
    # se amplió `epochs` en el YAML, hay que reanclarlo al total nuevo: se
    # devuelve el LR a lr0 y se adelanta el scheduler `start_epoch` pasos. Así
    # las épocas que faltan siguen exactamente el coseno de una corrida de
    # `epochs` épocas y terminan en LR ~0. Sin esto, el coseno viejo ya llegó a
    # su mínimo y las épocas extra entrenarían con LR ~0 (no aprenden nada) o,
    # peor, con el LR subiendo de nuevo hacia lr0 sobre el final.
    if scheduler is not None and start_epoch < training_cfg["epochs"]:
        for grupo in optimizer.param_groups:
            grupo["lr"] = training_cfg["lr0"]
            grupo.pop("initial_lr", None)
        scheduler = build_scheduler(optimizer, training_cfg, training_cfg["epochs"])
        for _ in range(start_epoch):
            scheduler.step()
        print(
            f"Scheduler reanclado a {training_cfg['epochs']} épocas. "
            f"LR de la próxima época: {optimizer.param_groups[0]['lr']:.6f}"
        )
else:
    print("No hay checkpoint previo. Entrenamiento desde cero.")

In [ ]:
# ============================================================
# Bucle de entrenamiento
# ============================================================

from src.engine.metrics import collect_predictions, compute_curves, compute_map
from src.engine.trainer import save_checkpoint, train_one_epoch

EPOCHS = training_cfg["epochs"]
elapsed_before = sum(row.get("epoch_time_min", 0.0) for row in history)
start_time = time.time()

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()
    print(f"\n{'=' * 70}\nÉpoca {epoch + 1}/{EPOCHS}\n{'=' * 70}")

    losses = train_one_epoch(model, optimizer, train_loader, DEVICE, scaler=scaler)

    if scheduler is not None:
        scheduler.step()

    predictions, targets = collect_predictions(model, val_loader, DEVICE)
    map_metrics = compute_map(predictions, targets)
    curves = compute_curves(predictions, targets)

    history.append(
        {
            "epoch": epoch + 1,
            "train/loss_total": losses["loss_total"],
            "train/loss_classifier": losses["loss_classifier"],
            "train/loss_box_reg": losses["loss_box_reg"],
            "train/loss_objectness": losses["loss_objectness"],
            "train/loss_rpn_box_reg": losses["loss_rpn_box_reg"],
            "metrics/mAP50": map_metrics["map50"],
            "metrics/mAP50-95": map_metrics["map50_95"],
            "metrics/precision": curves["best_precision"],
            "metrics/recall": curves["best_recall"],
            "lr": optimizer.param_groups[0]["lr"],
            "epoch_time_min": (time.time() - epoch_start) / 60,
        }
    )

    print(
        f"loss={losses['loss_total']:.4f} | "
        f"mAP50={map_metrics['map50']:.4f} | mAP50-95={map_metrics['map50_95']:.4f}"
    )

    # Se guarda al final de cada época: la sesión de Colab puede cortarse.
    save_checkpoint(CHECKPOINT_PATH, model, optimizer, scheduler, epoch + 1, history)
    print("Checkpoint guardado en:", CHECKPOINT_PATH)

train_time_min = elapsed_before + (time.time() - start_time) / 60
print(f"\nEntrenamiento finalizado. Tiempo total acumulado: {train_time_min:.1f} min")

In [ ]:
# ============================================================
# Guardar los pesos finales
# ============================================================

# Se guarda como last.pth y no best.pth: el bucle no hace seguimiento de la
# mejor época, así que estos son los pesos de la última, que son también los
# que evalúa el reporte de más abajo.
if experiment_config["output"]["save_weights"]:
    WEIGHTS_PATH = RUNS_DIR / experiment_name / "last.pth"
    torch.save(model.state_dict(), WEIGHTS_PATH)
    print("Pesos guardados en:", WEIGHTS_PATH)

In [ ]:
# ============================================================
# Reporte completo del experimento
# ============================================================

from src.reporting.experiment_report import generate_experiment_report

REPORTS_RESULTS_DIR = PROJECT_DIR / "reports" / "results" / experiment_name

metrics = generate_experiment_report(
    model=model,
    val_loader=val_loader,
    val_dataset=val_dataset,
    config=experiment_config,
    history=history,
    out_dir=REPORTS_RESULTS_DIR,
    device=DEVICE,
    train_time_min=train_time_min,
    device_name=DEVICE_NAME,
)

import pandas as pd
display(pd.DataFrame([metrics]).T.rename(columns={0: "valor"}))

In [ ]:
# ============================================================
# Visualización de las figuras generadas
# ============================================================

from IPython.display import Image, display

for nombre in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
]:
    ruta = REPORTS_RESULTS_DIR / nombre
    if ruta.exists():
        print(nombre)
        display(Image(filename=str(ruta)))
    else:
        print("No encontrado:", ruta)

In [ ]:
# ============================================================
# Publicación de resultados
# ============================================================
# En Colab y local se commitea directo al repo. En Kaggle no hay credenciales de
# git, así que se empaqueta todo y el commit se hace después desde la máquina
# donde sí están configuradas.

import subprocess

if IN_KAGGLE:
    entregable = Path("/kaggle/working") / f"{experiment_name}_resultados"
    if entregable.exists():
        shutil.rmtree(entregable)
    shutil.copytree(REPORTS_RESULTS_DIR, entregable / "reports_results")

    # last.pth va aparte del reporte: lo necesitan el notebook 05 y la demo, y
    # pesa demasiado para commitearlo al repo. El last_checkpoint.pth queda
    # fuera del zip a propósito: son 500 MB que Quick Save ya preserva como
    # output de la versión, y solo sirven para reanudar dentro de Kaggle.
    pesos = RUNS_DIR / experiment_name / "last.pth"
    if pesos.exists():
        shutil.copy(pesos, entregable / "last.pth")

    zip_path = shutil.make_archive(str(entregable), "zip", root_dir=entregable)
    shutil.rmtree(entregable)

    print("Paquete listo:", zip_path)
    print(f"Tamaño: {Path(zip_path).stat().st_size / 1e6:.1f} MB")
    print()
    print("Pasos siguientes:")
    print("  1. Save Version > Quick Save, para conservar la corrida completa.")
    print("  2. Bajar el zip desde el panel Output.")
    print(f"  3. Descomprimir reports_results/ en "
          f"reports/results/{experiment_name}/ del repo y commitear.")
    print(f"  4. Subir last.pth a Drive en VCII_DFire/runs/{experiment_name}/.")
else:
    %cd {PROJECT_DIR}

    !git config user.name "Gabriela-Sol"
    !git config user.email "solgab.salazar@gmail.com"

    pull_result = subprocess.run(
        ["git", "pull", "--rebase", "origin", REPO_BRANCH], text=True, capture_output=True
    )
    print(pull_result.stdout, pull_result.stderr)

    if pull_result.returncode != 0:
        raise RuntimeError("No se pudo completar git pull --rebase. Revisar conflictos.")

    for path in [
        f"reports/results/{experiment_name}/",
        "configs/experiments/fasterrcnn_r50fpn.yaml",
    ]:
        if Path(path).exists():
            subprocess.run(["git", "add", path], check=True)
            print("Agregado:", path)

    status = subprocess.run(["git", "status", "--short"], text=True, capture_output=True)
    print(status.stdout)

    if not status.stdout.strip():
        print("No hay cambios nuevos para commitear.")
    else:
        subprocess.run(
            ["git", "commit", "-m", f"results: update {experiment_name} outputs"], check=True
        )
        print(f"Commit creado. Para publicarlo: !git push origin {REPO_BRANCH}")